# 07 — Sentinel-5P (CDSE) acquisition plan (robust)

Creates a reproducible plan for acquiring Sentinel-5P (TROPOMI) L2 products via CDSE.

This notebook **does not download large data**; it prepares:
- a bounding box around the target site
- recommended products per pollutant
- provenance + QA rules

Supports config with **sites list** or **legacy dict**.

Outputs:
- `outputs/07_s5p_cdse_plan/s5p_acquisition_plan.json`
- `outputs/07_s5p_cdse_plan/manifest.json`


In [ ]:
import json, hashlib
from pathlib import Path
from datetime import datetime, timezone
import yaml

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def sha256_file(p: Path) -> str:
    return sha256_bytes(p.read_bytes())

def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")

def manifest_base(step: str, config_paths=None):
    m = {
        "step": step,
        "created_at_utc": utc_now(),
        "configs": [],
        "artifacts": []
    }
    for cp in (config_paths or []):
        if cp and cp.exists():
            m["configs"].append({"path": str(cp), "sha256": sha256_file(cp)})
    return m

def add_artifact(manifest: dict, path: Path, kind: str, meta=None):
    if path.exists():
        item = {"kind": kind, "path": str(path), "sha256": sha256_file(path)}
        if meta:
            item["meta"] = meta
        manifest["artifacts"].append(item)

# --- resolve repo root robustly ---
cwd = Path.cwd().resolve()

def find_repo_root(start: Path) -> Path:
    cur = start
    for _ in range(10):
        if (cur / "configs").exists() and (cur / "notebooks").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start

ROOT = find_repo_root(cwd)
CONFIGS = ROOT / "configs"
OUTPUTS = ROOT / "outputs"

print("UTC now:", utc_now())
print("ROOT:", ROOT)


In [ ]:
step = "07_s5p_cdse_plan"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

cfg_path = CONFIGS / "run.yml"
if not cfg_path.exists():
    raise FileNotFoundError("configs/run.yml not found")

cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8")) or {}

# defaults if satellite block not present
sat = cfg.get("satellite") or {}
bbox_km = float(sat.get("bbox_km", 30))  # default 30km box
qa_threshold = float(sat.get("qa_threshold", 0.5))

# normalize sites list/dict
sites = []
if isinstance(cfg.get("sites"), list):
    for s in cfg.get("sites", []):
        sites.append({
            "id": s.get("id") or s.get("site_id") or s.get("name"),
            "name": s.get("name"),
            "lat": s.get("lat"),
            "lon": s.get("lon"),
            "role": (s.get("role") or "context")
        })
elif isinstance(cfg.get("sites"), dict):
    for k, v in (cfg.get("sites") or {}).items():
        sites.append({
            "id": k,
            "name": v.get("name"),
            "lat": v.get("lat"),
            "lon": v.get("lon"),
            "role": (v.get("role") or "context")
        })

def pick_target_site(sites_list):
    for s in sites_list:
        if (s.get("role") or "").lower() == "target":
            return s
    for s in sites_list:
        nm = (s.get("name") or "").lower()
        if "newhaven" in nm or "erf" in nm or "inciner" in nm:
            return s
    return sites_list[0] if sites_list else None

target = pick_target_site(sites)
if not target:
    raise RuntimeError("No sites found in configs/run.yml")

lat, lon = float(target["lat"]), float(target["lon"])

# crude km->deg conversion (sufficient for bbox planning)
deg_lat = bbox_km / 111.0
deg_lon = bbox_km / (111.0 * max(0.2, abs(__import__('math').cos(lat * __import__('math').pi/180.0))))

bbox = {
    "min_lon": lon - deg_lon,
    "min_lat": lat - deg_lat,
    "max_lon": lon + deg_lon,
    "max_lat": lat + deg_lat,
}

plan = {
    "generated_at_utc": utc_now(),
    "target_site": {
        "id": target.get("id"),
        "name": target.get("name"),
        "lat": lat,
        "lon": lon,
        "role": target.get("role"),
    },
    "bbox_km": bbox_km,
    "bbox": bbox,
    "qa_threshold_default": qa_threshold,
    "products": {
        "NO2": {"product": "S5P_OFFL_L2__NO2____", "qa_field": "qa_value", "qa_min": qa_threshold},
        "SO2": {"product": "S5P_OFFL_L2__SO2____", "qa_field": "qa_value", "qa_min": qa_threshold},
        "CO":  {"product": "S5P_OFFL_L2__CO_____", "qa_field": "qa_value", "qa_min": qa_threshold},
        "HCHO": {"product": "S5P_OFFL_L2__HCHO___", "qa_field": "qa_value", "qa_min": qa_threshold},
        "O3":  {"product": "S5P_OFFL_L2__O3_____", "qa_field": "qa_value", "qa_min": qa_threshold},
        "CH4": {"product": "S5P_OFFL_L2__CH4____", "qa_field": "qa_value", "qa_min": qa_threshold},
        "AER_AI": {"product": "S5P_OFFL_L2__AER_AI_", "qa_field": "qa_value", "qa_min": qa_threshold}
    },
    "provenance_rules": [
        "store raw L2 files exactly as downloaded",
        "record product IDs + sensing start/end + orbit + processing baseline",
        "compute SHA256 for each raw file",
        "record exact query parameters + retrieval timestamp",
        "record QA mask used per pollutant"
    ],
    "interpretation_limits": [
        "regional-scale columns, not stack-level emissions",
        "use control comparisons + wind sector filtering"
    ]
}

write_json(out / "s5p_acquisition_plan.json", plan)

manifest = manifest_base(step, config_paths=[cfg_path])
add_artifact(manifest, out / "s5p_acquisition_plan.json", "plan")
write_json(out / "manifest.json", manifest)

print("Target:", target.get("id"), target.get("name"))
print("BBox:", bbox)
print("Wrote:", out / "manifest.json")
